# Operating-state classification with OYICD

Operating-state classification identifies how a machine is running rather than
whether it has failed. OYICD encodes one of eight modes in every recording name.

## Learning goals

- discover downloaded recordings without hard-coded paths;
- summarize each recording with Polars;
- classify modes while holding out later recordings.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "datasets").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import datasets
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score

In [ ]:
downloaded = datasets.download("oyicd")
files = sorted(Path(downloaded["extracted"]).rglob("*_mode*.csv"))
print(f"Recordings: {len(files)}")

def summarize_recording(path):
    frame = pl.read_csv(path)
    numeric = [
        column for column, dtype in frame.schema.items()
        if dtype.is_numeric()
    ]
    mode = int(re.search(r"mode(\d+)", path.name).group(1))
    return frame.select(
        *[pl.col(column).mean().alias(f"{column}_mean") for column in numeric],
        *[pl.col(column).std().alias(f"{column}_std") for column in numeric],
    ).with_columns(
        pl.lit(path.name).alias("recording"),
        pl.lit(mode).alias("mode"),
    )

recordings = pl.concat([summarize_recording(path) for path in files], how="diagonal_relaxed")
recordings.group_by("mode").len().sort("mode")

In [ ]:
recordings = recordings.with_columns(
    pl.int_range(pl.len()).over("mode").alias("mode_order"),
    pl.len().over("mode").alias("mode_count"),
)
train = recordings.filter(pl.col("mode_order") < pl.col("mode_count") * 0.8)
test = recordings.filter(pl.col("mode_order") >= pl.col("mode_count") * 0.8)
predictors = [
    column for column, dtype in recordings.schema.items()
    if dtype.is_numeric() and column not in {"mode", "mode_order", "mode_count"}
]

classifier = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=0,
)
classifier.fit(train.select(predictors).fill_null(0).to_numpy(), train["mode"].to_numpy())
prediction = classifier.predict(test.select(predictors).fill_null(0).to_numpy())
print(
    "Later-recording balanced accuracy:",
    f"{balanced_accuracy_score(test['mode'].to_numpy(), prediction):.3f}",
)

## Interpretation and limitations

Mode is an operating condition, not a fault label. Because recording time and
degradation may be correlated, a chronological split is preferable to random
rows. A production model should also test whether mode recognition transfers
between machines and component ages.